# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** James Emmanuel Yeboah-Asiamah
**Student ID:** 14022028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [20]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GEMINI_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",   # remove this line if using OpenAI itself
)
MODEL = "gemma-4-31b-it"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [21]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.", temperature=0.7, max_tokens=300):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
],
temperature=temperature,
max_tokens=max_tokens,
)
    return response
#
# TODO: Call it once with a simple question and print the answer.
test_response = ask_llm("What are three key principles of responsible financial lending?")
print(f"Answer: {test_response.choices[0].message.content}")
# TODO: Print response.usage as well — how many tokens did your call consume?
print("\nToken Usage Details:")
print(test_response.usage.total_tokens)
print(f"Prompt tokens:     {test_response.usage.prompt_tokens}")
print(f"Completion tokens: {test_response.usage.completion_tokens}")

Answer: <thought>Responsible financial lending.
Three key principles.

    *   *What makes lending "responsible"?* It's not just about making money; it's about ensuring the borrower can pay it back without falling into a debt trap, protecting the lender's assets, and adhering to legal/ethical standards.
    *   *Brainstorming keywords:* Creditworthiness, transparency, affordability, fair interest, consumer protection, sustainability, ethics, risk management.

    *   *Principle 1: Assessment of Ability to Repay (Affordability).* This is the cornerstone. If the borrower can't pay, it's predatory or negligent.
    *   *Principle 2: Transparency and Fairness (Disclosure).* The borrower needs to know exactly what they are signing up for (rates, fees, terms).
    *   *Principle 3: Ethical Treatment and Consumer Protection (Fair Practices).* How the lender behaves during the process and during collection.

    *   *Principle 1: Affordability/Creditworthiness.*
        *   Focus: Due diligenc

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** [
    1) The system role defines the model's overall persona, instructions, tone and acts as the background operational context. Example (you are a risk-averse microfinance credit analyst in Ghana. Summarise loan requests factually without inventing unstated details)
    User roles contains the specific instruction/task or input data that the model should process during the current interaction with the user. Example (Summarise loan requests factually without inventing unstated details)
]
###
> [
    2) A token is a foundational unit of text processed by a model in natural language processing. API provider bill per token because the computational cost (GPU, energy and compute time) scale based on the length of the input and output text rather than the raw number of HHTP requests. Since the input context and output length can be reflected in the number of tokens obtained from the input and generated for the output, it makes sense to charge per token
]

### Part 1.2 — Temperature: the randomness dial

In [26]:
import time
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
print("------ 5 results of the prompt, 'What are three key principles of responsible financial lending?' at temperature = 0.0 ------ ")
for i in range(5):
    test_response1 = ask_llm("What are three key principles of responsible financial lending?", temperature= 0.0)
    print(f" Answer{i+1}: {test_response1.choices[0].message.content}")

    time.sleep(5)

print("------ 5 results of the prompt, 'What are three key principles of responsible financial lending?' at temperature = 1.2 ------")
for i in range(5):
    test_response2 = ask_llm("What are three key principles of responsible financial lending?", temperature= 1.2)
    print(f" Answer{i+1}: {test_response2.choices[0].message.content}")

    time.sleep(5)

# TODO: Print all 10 answers, grouped by temperature.

------ 5 results of the prompt, 'What are three key principles of responsible financial lending?' at temperature = 0.0 ------ 
 Answer1: <thought>Responsible financial lending.
Three key principles.
Provide a clear, professional, and comprehensive explanation of these principles.

    *   What makes lending "responsible"?
    *   It's not just about making money; it's about the borrower's ability to pay and the lender's ethical obligations.
    *   *Key concepts:* Creditworthiness, transparency, fairness, sustainability, consumer protection, risk management.

    *   *Principle 1: Affordability/Creditworthiness.* (The lender shouldn't lend more than the person can pay back).
    *   *Principle 2: Transparency/Disclosure.* (The borrower should know exactly what they are signing up for—rates, fees, terms).
    *   *Principle 3: Fairness/Ethical Treatment.* (No predatory practices, fair collection methods, non-discrimination).

    *   *Principle 1: Affordability (The "Ability to Repay").

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** [At temperature 0.0, there was no variation in all 5 responses. They were identical in wording because. At temperature 1.2, there was significant variation in the 5 responses from the model. Different wordings were used in each response. A low temperature regime (0.0 to maybe 0.2) is appropriate in this case because finances are a high-stakes domain and thus advice and accuracy must be high and stable. For that reason, the model cannot be having constantly varying responses. We need to produce consistent responses]

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [27]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [43]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this: \n\n {letter_text}"

print("====== V1 Output on L002 ======")
res_v1_l002 = ask_llm(SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"]))
print(res_v1_l002.choices[0].message.content)
time.sleep(5)

print("====== V1 Output on L006 ======")
res_v1_l006 = ask_llm(SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"]))
print(res_v1_l006.choices[0].message.content)
time.sleep(5)
# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_PROMPT_V2 = "You are an assistant to a microfinance loan officer evaluating small businessses in Ghana. Summarize this for him in a 3-4 sentences brief for him: \n\n {letter_text}. In the summary, use neutral, clear language and do not add any invented details."

print("====== V2 Output on L002 (temperature = 0.0)======")
res_v2_l002 = ask_llm(SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"]),
                      system_prompt=SUMMARY_PROMPT_V2,
                      temperature=0.0)

print(res_v2_l002.choices[0].message.content)
time.sleep(5)

print("=================================================================================================")

print("=== V2 Output on L006 (temperature = 0.0) ===")
res_v2_l006 = ask_llm(SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"]),
                      system_prompt= SUMMARY_PROMPT_V2,
                      temperature= 0.0)

print(res_v2_l006.choices[0].message.content)
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

====== V1 Output on L002 ======
<thought>
*   Source: A short message from Kwame Boateng.
*   Key Information:
    *   Name: Kwame Boateng.
    *   Profession: Commercial driver in Kumasi.
    *   Amount requested: GHS 25,000.
    *   Purpose: Repair trotro engine and settle personal debts.
    *   Current situation: Slow business, expecting improvement after the festive season.
    *   Repayment plan: Vague ("whenever the money comes").
    *   Collateral: None.
    *   Tone: Urgent, hopeful/religious.

    *   Who: Kwame Boateng, a commercial driver from Kumasi.
    *   What: Requesting GHS 25,000.
    *   Why: Engine repairs and debt settlement.
    *   Terms: No collateral, uncertain repayment date.

    *   *Option 1 (One sentence/Concise):* Kwame Boateng, a commercial driver in Kumasi, is urgently requesting GHS 25,000 for engine repairs and debt settlement, offering no collateral and an unspecified repayment timeline.
    *   *Option 2 (Bullet points):*
        *   Requestor: Kw

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** [1) V1 was unable to follow the length constraints The outputs were getting cut off mid-thought (e.g., stopping abruptly at "Purpose: Trotro engine repair" on L002 and "a provision shop, and a" on L006), failing to produce a finalized, usable final response. Also, V1 kept generating long branching monologues with multiple optional outputs (Option 1, Option 2, Option 3) instead of a single summary. This means it's thrown decision-making back to the officer to decide which one which defeats the point of the summary. But V2 adhered to the constraints and produced a concise summary]
##
[2) Finances are a high-stake domain because it affects people's livelihoods. If an LLM invents unstated income figures, a loan officer could end up making unjustified credit approval based on false data. This failure mode is called hallucination]

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [50]:
import json
import re
import time
import pandas as pd
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_PROMPT = """You're a strict data extraction assistant. Extract structured financial data from loan application letters into JSON format.
The output MUST be a single valid JSON object with EXACTLY these keys:
- "applicant_name":string(full name or first name as given),
- "amount_ghs": number (GHS amount requested in numbers),
- "purpose": string (short description of how the money will be used),
- "monthly_profit_ghs": number or null (monthly profit in GHS; use null if unstated),
- "has_collateral_or_guarantor": boolean (true if applicant offers collateral, fixed deposit, group guarantee, or individual guarantor; false otherwise),
- "repayment_months": number or null (proposed term in months; use null if unstated)

CRITICAL RULES:
1. Output NOTHING except valid raw JSON. Do NOT include <thought> tags, reasoning, explanation, or markdown formatting.
2. Begin your response directly with the opening curly brace '{'.
2. If a field is not explicitly stated in the letter, set its value to null. Do NOT guess or infer missing values.
Example application:
'Good day, I am Baba Seidu, a carpenter in Tamale. I need GHS 5,000 for timber stock. I make about GHS 1,200 profit monthly. I have no guarantor or collateral. I will pay back over 10 months.'
Example JSON Output:
{
  "applicant_name": "Baba Seidu",
  "amount_ghs": 5000,
  "purpose": "timber stock",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": false,
  "repayment_months": 10
}
"""
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    response = ask_llm(user_prompt= f"Application Letter: \n{letter_text}", system_prompt=EXTRACT_PROMPT, temperature= 0.0, max_tokens= 600)
    raw_response = response.choices[0].message.content.strip()

    cleaned_response = re.sub(r"<thought>.*?</thought>", "", raw_response, flags=re.DOTALL).strip()
    cleaned_response = re.sub(r"^```(?:json)?\s*", "", cleaned_response, flags=re.MULTILINE)
    cleaned_response = re.sub(r"\s*```$", "", cleaned_response, flags=re.MULTILINE)


    json_match = re.search(r"\{.*\}", cleaned_response, flags=re.DOTALL)
    if json_match:
        cleaned_response = json_match.group(0)

    try:
        data = json.loads(cleaned_response)
        return data
    except Exception as e:
        print(f"WARNING: Failed to parse JSON response. Error: {e}")
        print(f"Raw response was:\n{raw_response}")
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
extracted_results = {}
for code, text in LETTERS.items():
    extracted_results[code] = extract_fields(text)
    time.sleep(5)

df_extracted = pd.DataFrame.from_dict(extracted_results, orient="index")
display(df_extracted)

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [1) Using letters from the six letters will cause data leakage. The model would see the actual data and labels in its prompt context, which would prevent us from truly knowing whether the correct output was because of the quality of the prompting or it was from the model knowing exactly the data content]
##
[2) Without explicit instruction, the model hallucinated missing numeric values for labels such as monthly profit for Kofi in L006]
##
[3)Extraction is a deterministic task with a single ground truth. Temperature=0 forces greedy decoding to consistently select the most probable correct token. For creative tasks, zero temperature would produce consistent, monotonous and repetitive text.]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [51]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_SYSTEM = """You are a senior microfinance credit analyst preparing a decision-support brief for a human loan officer.
You must synthesize the original letter and extracted JSON data.

Your output must follow this format:
### 1. Strengths
- [bullet points]

### 2. Risks / Red Flags
- [bullet points]

### 3. Missing Information
- [bullet points]

### 4. Suggested Next Step
[Provide a clear, actionable recommendation such as 'Invite applicant for interview', 'Request supporting documents', 'Flag for senior risk committee review', or 'Decline consideration at preliminary stage'.]

IMPORTANT CONSTRAINT: You are a Decision Support Tool. Final credit decisions are exclusively made by human loan officers. DO NOT issue a final binding 'Approve' or 'Reject' decision."""

BRIEF_USER = """Loan Application ID: {letter_id}

Extracted JSON Data:
{json_data}

Original Application Letter:
{letter_text}

Generate the decision-support brief:"""

def generate_brief(letter_id, letter_text, extracted_dict):
    json_str = json.dumps(extracted_dict, indent=2)
    prompt = BRIEF_USER.format(
        letter_id=letter_id, json_data=json_str, letter_text=letter_text
    )
    res = ask_llm(
        prompt, system_prompt=BRIEF_SYSTEM, temperature=0.1, max_tokens=600
    )
    return res.choices[0].message.content

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
briefs = {}
for code in LETTERS:
    briefs[code] = generate_brief(
        code, LETTERS[code], extracted_results[code]
    )

for target_code in ["L001", "L002", "L006"]:
    print(f"\n==================== BRIEF FOR {target_code} ====================")
    print(briefs[target_code])


==================== BRIEF FOR L001 ====================
<thought>Senior Microfinance Credit Analyst.
Human Loan Officer.
Synthesize a loan application letter and extracted JSON data into a decision-support brief.

        *   ### 1. Strengths
        *   ### 2. Risks / Red Flags
        *   ### 3. Missing Information
        *   ### 4. Suggested Next Step
Do NOT make a final binding decision (Approve/Reject).

    *   *Applicant:* Akosua Mensah.
    *   *Loan Amount:* GHS 8,000.
    *   *Purpose:* Deep freezer/frozen foods expansion.
    *   *Monthly Profit:* GHS 900.
    *   *Repayment Plan:* GHS 450/month for 20 months.
    *   *Collateral/Guarantor:* Sister (Teacher).
    *   *Experience:* 12 years selling provisions at Makola Market.
    *   *Internal History:* GHS 2,500 saved in susu scheme; perfect contribution record for 2 years.

    *   *Strengths:*
        *   Long-term business stability (12 years at Makola Market).
        *   Proven repayment behavior/discipline (perfect

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [1) L003 (Efua Darko): The system highlighted strong factors which registered business history, GCB fixed deposit collateral, strong seasonal cash flow (GHS 22,000 December revenue), and clear capital growth plan. The displeasing factors were minor (e.g., dependence on Christmas peak season). L006 (Kofi): The system flagged major risk factors which were zero business experience, no collateral/guarantor, unstated profit, and a fragmented business proposal (car wash + provision shop + Dubai phone imports)]
##
[2) Practical Reason: LLMs lack complete real-world context such as local market conditions and physical site visits and thus, can be led astray by persuasive writing. Ethical Reason: Fully automated rejections remove human accountability and create algorithmic fairness risks. A human loan officer must evaluate and remain legally accountable for all credit decisions made by the system.]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [bca4a6e2aa20cf47339997ba9782a56c1fea3c23]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [52]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]
evaluation_data = {field: {} for field in fields}

for code in ["L001", "L003", "L006"]:
    gold_dict = GOLD[code]
    predicted_dictionary = extracted_results[code]

    for field in fields:
        g_val = gold_dict[field]
        p_val = predicted_dictionary.get(field)

        if field == "applicant_name":
            match = str(g_val).lower() in str(p_val).lower()
        elif field == "purpose":
            match = True
        else:
            match = g_val == p_val

        evaluation_data[field][code] = "MATCH" if match else f"MISMATCH ({p_val})"
# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
for field in fields:
    matches = sum(1 for code in ["L001", "L003", "L006"] if "MATCH" in evaluation_data[field][code])
    evaluation_data[field]["Accuracy"] = f"{matches/3:.0%}"

df_eval = pd.DataFrame(evaluation_data).T
display(df_eval)

,L001,L003,L006,Accuracy
applicant_name,MATCH,MATCH,MATCH,100%
amount_ghs,MATCH,MATCH,MATCH,100%
purpose,MATCH,MATCH,MATCH,100%
monthly_profit_ghs,MATCH,MATCH,MATCH,100%
has_collateral_or_guarantor,MATCH,MATCH,MATCH,100%
repayment_months,MATCH,MATCH,MATCH,100%


### Part 4.2 — Reliability: is the system consistent?

In [55]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

def test_reliability(letter_text, temperature, runs=5):
    valid_json_count = 0
    unique_outputs = set()

    for _ in range(runs):
        response = ask_llm(user_prompt= f"Application Letter: \n{letter_text}", system_prompt=EXTRACT_PROMPT, temperature= 0.0, max_tokens= 600)
        raw_response = response.choices[0].message.content.strip()
        cleaned_response = re.sub(r"<thought>.*?</thought>", "", raw_response, flags=re.DOTALL).strip()
        cleaned_response = re.sub(r"^```(?:json)?\s*", "", cleaned_response, flags=re.MULTILINE)
        cleaned_response = re.sub(r"\s*```$", "", cleaned_response, flags=re.MULTILINE)
        
        
        json_match = re.search(r"\{.*\}", cleaned_response, flags=re.DOTALL)
        if json_match:
            cleaned_response = json_match.group(0)
        
        try:
            data = json.loads(cleaned_response)
            valid_json_count += 1
            unique_outputs.add(json.dumps(data, sort_keys=True))
        except Exception:
                pass 

    return valid_json_count, len(unique_outputs)


v_t0, u_t0 = test_reliability(LETTERS["L004"], temperature=0.0)
v_t1, u_t1 = test_reliability(LETTERS["L004"], temperature=1.0)

print(f"Temperature 0.0: {v_t0}/5 valid JSON, {u_t0} unique variation(s) across runs.")
print(f"Temperature 1.0: {v_t1}/5 valid JSON, {u_t1} unique variation(s) across runs.")

Temperature 0.0: 5/5 valid JSON, 1 unique variation(s) across runs.
Temperature 1.0: 5/5 valid JSON, 1 unique variation(s) across runs.


### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.